# Senescence Agent — Full Backend on Google Colab (Zero Cost)

This notebook runs the **entire backend** (FastAPI + Gemini agent + Scanpy tools) on Google Colab for free.

**What you need before starting:**
1. A free Gemini API key from https://aistudio.google.com/apikey
2. A `.h5ad` dataset uploaded to Google Drive (see Step 2 below)

**Runtime:** CPU is fine. No GPU needed. Scanpy runs on CPU.

---
## Step 1: Mount Google Drive

**Why:** Your `.h5ad` datasets live on Drive so they persist across Colab sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## Step 2: Download a Dataset (Run Once)

**Why:** The agent analyzes single-cell `.h5ad` files. You need at least one.

**Option A — If you already uploaded a .h5ad to Drive, skip this cell.**

**Option B — Download Tabula Muris Senis programmatically:**

In [ ]:
import os

DRIVE_DATA = "/content/drive/MyDrive/senescence-agent-data"
os.makedirs(DRIVE_DATA, exist_ok=True)

KIDNEY_PATH = os.path.join(DRIVE_DATA, "tms_kidney.h5ad")

if os.path.exists(KIDNEY_PATH):
    print(f"Dataset already exists: {KIDNEY_PATH}")
    print(f"Size: {os.path.getsize(KIDNEY_PATH) / 1e6:.1f} MB")
else:
    print("Dataset not found on Drive.")
    print("")
    print("=== HOW TO GET THE DATASET ===")
    print("")
    print("Option 1: Download from CellxGene (recommended)")
    print("  1. Go to: https://cellxgene.cziscience.com/collections/0b9d8a04-bb9d-44da-aa27-705bb65b54eb")
    print("  2. Find 'Kidney' tissue, click the download icon, choose '.h5ad'")
    print("  3. Upload the downloaded file to Google Drive at:")
    print(f"     {DRIVE_DATA}/tms_kidney.h5ad")
    print("")
    print("Option 2: Download from Figshare")
    print("  1. Go to: https://figshare.com/projects/Tabula_Muris_Senis/64982")
    print("  2. Find the FACS Kidney .h5ad file")
    print("  3. Upload to Drive at the same path above")
    print("")
    print("Option 3: Use the test PBMC dataset (small, 700 cells — good for testing)")
    print("  Run the next cell to download it automatically.")

In [ ]:
# Option 3: Download small PBMC test dataset (700 cells, ~2 MB)
# Good for testing the pipeline, but has NO age column (age comparison won't work)

import scanpy as sc

PBMC_PATH = os.path.join(DRIVE_DATA, "pbmc_test.h5ad")

if not os.path.exists(PBMC_PATH):
    print("Downloading PBMC test dataset...")
    adata = sc.datasets.pbmc68k_reduced()
    adata.write_h5ad(PBMC_PATH)
    print(f"Saved: {PBMC_PATH} ({adata.shape[0]} cells, {adata.shape[1]} genes)")
else:
    print(f"Already exists: {PBMC_PATH}")

---
## Step 3: Clone the Repo & Install Dependencies

**Why:** Gets the agent code and all Python packages (~3-5 min first time).

In [ ]:
# Clone repo (or pull latest if already cloned)
if os.path.exists("/content/senescence-agent"):
    !cd /content/senescence-agent && git pull
else:
    !git clone https://github.com/Ro-netizen004/senescence-agent.git /content/senescence-agent

print("Repo ready.")

In [ ]:
# Install Python dependencies (3-5 minutes first time, cached after)
!pip install -q \
    fastapi==0.136.1 \
    uvicorn==0.46.0 \
    python-multipart==0.0.28 \
    python-dotenv==1.2.2 \
    "google-generativeai>=0.8.3" \
    anndata==0.11.4 \
    scanpy==1.11.5 \
    leidenalg==0.11.0 \
    "pydeseq2>=0.4.4" \
    mygene==3.2.2 \
    pyngrok \
    nest_asyncio

print("\nAll packages installed.")

---
## Step 4: Set Your Gemini API Key

**Why:** The agent uses Gemini (free tier: 15 req/min, 1M tokens/day) to route tool calls.

**How to get a key:**
1. Go to https://aistudio.google.com/apikey
2. Click "Create API Key"
3. Copy it and paste below

In [ ]:
# ==============================
# PASTE YOUR GEMINI API KEY HERE
# ==============================
GEMINI_API_KEY = ""  # <-- paste between the quotes

# Validate
assert GEMINI_API_KEY and len(GEMINI_API_KEY) > 10, "Please paste your Gemini API key above!"

# Write .env file
with open("/content/senescence-agent/.env", "w") as f:
    f.write(f"GEMINI_API_KEY={GEMINI_API_KEY}\n")
    f.write("GEMINI_MODEL=gemini-2.5-flash\n")

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["GEMINI_MODEL"] = "gemini-2.5-flash"

# Quick test
import google.generativeai as genai
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")
resp = model.generate_content("Say OK if you can hear me. One word only.")
print(f"Gemini says: {resp.text.strip()}")
print("API key is working!")

---
## Step 5: Test the Agent Directly (No Server Needed)

**Why:** Verify the full pipeline works before starting the server. This runs the agent loop in Python — same code the server uses.

In [ ]:
import sys
sys.path.insert(0, "/content/senescence-agent/backend")
os.chdir("/content/senescence-agent/backend")

# This import triggers SenMayo gene conversion (~30 sec first time)
print("Loading agent modules (gene conversion may take 30 sec)...")
from agent.agent import run_agent
from agent.cache import cache_adata
import scanpy as sc

print("Agent modules loaded.")

In [ ]:
# Choose your dataset
# Uncomment ONE of these:

DATASET_PATH = os.path.join(DRIVE_DATA, "tms_kidney.h5ad")    # TMS Kidney (full demo)
# DATASET_PATH = os.path.join(DRIVE_DATA, "pbmc_test.h5ad")   # PBMC (quick test, no age column)

SPECIES = "mouse"  # "mouse" or "human"

# Load
print(f"Loading {DATASET_PATH}...")
adata = sc.read_h5ad(DATASET_PATH)
print(f"Loaded: {adata.shape[0]} cells, {adata.shape[1]} genes")
print(f"Columns: {list(adata.obs.columns)[:15]}")

# Cache it (same as what /upload does)
FILE_ID = "demo-kidney"
cache_adata(FILE_ID, adata)
print(f"\nCached as file_id='{FILE_ID}'")

In [ ]:
# Run the full senescence analysis panel
print("Running full senescence analysis (this takes 30-60 sec)...")
print("="*60)

result = run_agent(
    session_history=[],
    message="Run the full senescence analysis",
    file_id=FILE_ID,
    species=SPECIES,
)

print("\n" + "="*60)
print("TOOLS CALLED:")
for tc in result["tool_calls"]:
    print(f"  - {tc['name']}")

print(f"\nPLOTS GENERATED: {len(result['plots'])}")
for p in result["plots"]:
    print(f"  - {p['url']}")

print(f"\nRESPONSE (first 1000 chars):")
print(result["reply"][:1000])

In [ ]:
# Display the generated plots
from IPython.display import Image, display
import glob

output_dir = "/content/senescence-agent/backend/outputs"
plots = glob.glob(os.path.join(output_dir, "*.png"))

for plot_path in sorted(plots):
    print(f"\n--- {os.path.basename(plot_path)} ---")
    display(Image(filename=plot_path, width=600))

In [ ]:
# Test a statistical question (only works with TMS data that has age column)
print("Testing: p-value for T cells, 3m vs 24m...")
print("="*60)

result2 = run_agent(
    session_history=[],
    message="What is the p-value for senescence difference in T cells, 3m vs 24m?",
    file_id=FILE_ID,
    species=SPECIES,
)

print(result2["reply"][:1000])
print(f"\nTools: {[tc['name'] for tc in result2['tool_calls']]}")

---
## Step 6: Start the FastAPI Server (For Frontend Connection)

**Why:** The React frontend talks to the backend via HTTP. We use ngrok to create a public URL that the frontend can reach.

**Note:** You need a free ngrok account. Sign up at https://ngrok.com and get your auth token from the dashboard.

In [ ]:
# ==============================
# PASTE YOUR NGROK AUTH TOKEN HERE (free signup at ngrok.com)
# Go to: https://dashboard.ngrok.com/get-started/your-authtoken
# ==============================
NGROK_TOKEN = ""  # <-- paste between the quotes

if NGROK_TOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)
    print("ngrok configured.")
else:
    print("No ngrok token — server will only be accessible within Colab.")
    print("You can still test the agent directly in Step 5 above.")

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import subprocess, time

# Copy dataset to uploads folder so /upload isn't needed for testing
uploads_dir = "/content/senescence-agent/backend/data/uploads"
os.makedirs(uploads_dir, exist_ok=True)

import shutil
demo_upload = os.path.join(uploads_dir, f"{FILE_ID}.h5ad")
if not os.path.exists(demo_upload):
    shutil.copy2(DATASET_PATH, demo_upload)
    print(f"Copied dataset to {demo_upload}")

# Start the FastAPI server
proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/senescence-agent/backend",
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

time.sleep(8)  # Wait for server startup + gene conversion

# Check if server is running
import urllib.request
try:
    resp = urllib.request.urlopen("http://localhost:8000/health")
    print(f"Server health: {resp.read().decode()}")
except Exception as e:
    print(f"Server may still be starting... ({e})")
    print("Wait 30 sec and re-run this cell if needed.")

In [ ]:
# Create public URL via ngrok
if NGROK_TOKEN:
    from pyngrok import ngrok
    public_url = ngrok.connect(8000)
    BACKEND_URL = str(public_url).replace('NgrokTunnel: "', '').rstrip('"').split('"')[0]
    # Clean up the URL
    if ' ' in BACKEND_URL:
        BACKEND_URL = BACKEND_URL.split('"')[0]
    print("\n" + "="*60)
    print(f"BACKEND URL: {BACKEND_URL}")
    print("="*60)
    print(f"\nHealth check:  {BACKEND_URL}/health")
    print(f"\nFor frontend .env file, set:")
    print(f"  VITE_API_URL={BACKEND_URL}")
    print(f"\nThis URL expires in ~2 hours. Re-run this cell to get a new one.")
else:
    print("No ngrok token. Server running at http://localhost:8000 (Colab-only).")
    print("Use Step 5 to test the agent directly in Python.")

---
## Step 7: Test Server with curl (Optional)

**Why:** Verify the HTTP endpoints work before connecting the frontend.

In [ ]:
import json, urllib.request

# Test /chat endpoint directly
payload = json.dumps({
    "session_id": "colab-test",
    "message": "Score the cells for senescence",
    "file_id": FILE_ID,
    "species": SPECIES,
}).encode()

req = urllib.request.Request(
    "http://localhost:8000/chat",
    data=payload,
    headers={"Content-Type": "application/json"},
)

print("Sending chat request (may take 20-30 sec)...")
resp = urllib.request.urlopen(req, timeout=120)
data = json.loads(resp.read().decode())

print(f"\nTools called: {[t['name'] for t in data.get('tool_calls', [])]}")
print(f"Plots: {[p['url'] for p in data.get('plots', [])]}")
print(f"\nReply (first 500 chars):\n{data['reply'][:500]}")

---
## Step 8: Generate a Report

**Why:** Tests the /report endpoint that produces Markdown + PDF reports.

In [ ]:
# Generate report from previous tool runs
report_payload = json.dumps({
    "session_id": "colab-test",
    "file_id": FILE_ID,
    "species": SPECIES,
    "tool_runs": result["tool_calls"],  # from Step 5
    "plots": result["plots"],
}).encode()

req = urllib.request.Request(
    "http://localhost:8000/report",
    data=report_payload,
    headers={"Content-Type": "application/json"},
)

print("Generating report (may take 15-30 sec)...")
resp = urllib.request.urlopen(req, timeout=120)
report_data = json.loads(resp.read().decode())

print(f"Report URL: {report_data.get('report_url')}")
print(f"PDF URL: {report_data.get('pdf_url')}")
print(f"\nReport preview (first 800 chars):\n")
print(report_data.get('report', '')[:800])

---
## Step 9: Run Validation Analysis (GSE226225 — For Presentation Slides)

**Why:** This produces the numbers for your strongest slide: "Our tool identified X% of labeled senescent cells on a held-out dataset."

**Prerequisite:** Download GSE226225 and upload to Drive. See docs/validation_and_comparison.md for details.

In [ ]:
# Uncomment and run when you have the GSE226225 dataset
"""
import numpy as np

val_path = os.path.join(DRIVE_DATA, "GSE226225.h5ad")
val_adata = sc.read_h5ad(val_path)
print(f"Validation dataset: {val_adata.shape[0]} cells, {val_adata.shape[1]} genes")
print(f"Columns: {list(val_adata.obs.columns)}")

# Look for the senescence label column
# (print columns above, then set the correct one here)
# SENESCENCE_COL = "senescence_label"  # adjust based on actual column name
"""

---
## Step 10: Connect the Frontend

**Option A — Run frontend on your laptop (recommended for demo):**
```bash
cd frontend
npm install
echo "VITE_API_URL=YOUR_NGROK_URL" > .env
npm run dev
```
Then open http://localhost:5173

**Option B — Build and deploy to Netlify (free):**
```bash
cd frontend
echo "VITE_API_URL=YOUR_NGROK_URL" > .env
npm run build
# Drag 'dist' folder to netlify.com/drop
```

---
## Troubleshooting

| Problem | Solution |
|---------|----------|
| "ModuleNotFoundError" | Re-run the pip install cell |
| "GEMINI_API_KEY not configured" | Re-run the API key cell |
| "Dataset not found" | Check the file path, re-run the cache cell |
| Server timeout | Gene conversion takes ~30 sec on first load, wait and retry |
| Out of memory | Subsample: `sc.pp.subsample(adata, n_obs=5000)` |
| Colab disconnected | Re-run all cells from top (Drive data persists) |
| ngrok expired | Re-run the ngrok cell to get new URL |
| Gemini rate limit | Wait 60 sec between rapid requests (free tier: 15/min) |